# Selección de columnas y control de data leakage

Este notebook corresponde a la issue #8. Define qué columnas entran en el modelo y cuáles se conservan únicamente para visualización o se eliminan por privacidad, identificabilidad o leakage.

Las coordenadas se mantienen tanto para el mapa como para el modelo. No se crean dos ficheros: se generan dos vistas en memoria desde la misma fuente.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path("..").resolve()))
from src.cleaning import (
    MODEL_EXCLUDED_COLUMNS,
    clean_listings,
    select_map_view,
    select_model_features,
)

DATA_PATH = Path("../data/raw/listings.csv.gz")
raw_df = pd.read_csv(DATA_PATH, compression="gzip")
cleaned_df = clean_listings(raw_df)
model_df = select_model_features(cleaned_df)
map_df = select_map_view(cleaned_df)

print(f"Dataset limpio: {cleaned_df.shape[0]:,} filas x {cleaned_df.shape[1]:,} columnas")
print(f"Vista del modelo: {model_df.shape[0]:,} filas x {model_df.shape[1]:,} columnas")
print(f"Vista del mapa: {map_df.shape[0]:,} filas x {map_df.shape[1]:,} columnas")

Dataset limpio: 12,610 filas x 93 columnas
Vista del modelo: 12,610 filas x 55 columnas
Vista del mapa: 12,610 filas x 5 columnas


## Columnas excluidas del modelo

In [2]:
available_excluded = [column for column in MODEL_EXCLUDED_COLUMNS if column in cleaned_df.columns]
reasons = {
    "id": "identificador técnico",
    "listing_url": "URL del anuncio",
    "scrape_id": "identificador del snapshot",
    "name": "texto libre del anuncio",
    "host_name": "nombre del anfitrión",
    "host_about": "texto libre potencialmente personal",
    "host_id": "identificador del anfitrión",
    "host_location": "localización del anfitrión",
    "license": "identificador legal",
    "neighbourhood": "columna completamente vacía",
    "neighborhood_overview": "columna completamente vacía",
    "host_since": "columna completamente vacía; se usan variables derivadas",
    "host_response_time": "columna completamente vacía",
    "host_response_rate": "columna completamente vacía",
    "host_acceptance_rate": "columna completamente vacía",
    "host_total_listings_count": "columna completamente vacía",
    "host_verifications": "columna completamente vacía",
    "bathrooms": "columna completamente vacía; se usará bathrooms_text",
    "instant_bookable": "columna completamente vacía",
    "source": "75,75 % nulo y un único valor",
    "description": "texto libre de alta cardinalidad",
    "price": "fuente del target",
    "estimated_revenue_l365d": "derivada de ingresos",
    "price_quote_total_price": "derivada de precio",
    "price_quote_price_per_night": "derivada de precio",
}
selection_audit = pd.DataFrame({
    "column": sorted(available_excluded),
    "reason": [reasons.get(column, "metadata o posible leakage") for column in sorted(available_excluded)],
})
display(selection_audit)

,column,reason
0,bathrooms,columna completamente vacía; se usará bathroom...
1,calendar_last_scraped,metadata o posible leakage
2,calendar_updated,metadata o posible leakage
3,description,texto libre de alta cardinalidad
4,estimated_revenue_l365d,derivada de ingresos
5,host_about,texto libre potencialmente personal
6,host_acceptance_rate,columna completamente vacía
7,host_id,identificador del anfitrión
8,host_location,localización del anfitrión
9,host_name,nombre del anfitrión


## Dos vistas desde una única fuente

In [3]:
print("Columnas de la vista del mapa:")
display(map_df.head())

print("¿Las coordenadas entran en el modelo?", {"latitude", "longitude"}.issubset(model_df.columns))
print("¿Las coordenadas están disponibles para el mapa?", {"latitude", "longitude"}.issubset(map_df.columns))
print("¿Price entra como feature?", "price" in model_df.columns)

Columnas de la vista del mapa:


,latitude,longitude,neighbourhood_cleansed,neighbourhood_group_cleansed,price
0,41.40556,2.17262,la Sagrada Família,Eixample,274.0
1,41.41243,2.21975,el Besòs i el Maresme,Sant Martí,407.67
2,41.37978,2.17623,el Barri Gòtic,Ciutat Vella,545.6
3,41.38043,2.19094,la Barceloneta,Ciutat Vella,44.47
4,41.39631,2.16832,la Dreta de l'Eixample,Eixample,187.46


¿Las coordenadas entran en el modelo? True
¿Las coordenadas están disponibles para el mapa? True
¿Price entra como feature? False


## Decisiones

- Se eliminan identificadores técnicos, URLs, nombres y textos potencialmente personales.
- Se eliminan variables derivadas directamente de precio o ingresos.
- `price` se conserva en `cleaned_df` para crear el target, pero se excluye de `model_df`.
- `latitude` y `longitude` se conservan en `model_df` y `map_df`; se comparará su aportación con una versión del modelo sin coordenadas.
- `neighbourhood_cleansed` y `neighbourhood_group_cleansed` permanecen como variables geográficas interpretables.
- La codificación de las variables categóricas se realizará posteriormente dentro del pipeline de #13.